# Feature Engineering — Delivery Delay Prediction

Joins deliveries with vehicle + route attributes and derives pre-departure
features for delay classification. EXCLUDES post-trip leakage
(actual_arrival, actual_duration_hrs, delay_hrs, status).

**Reads:** `silver_deliveries`, `silver_vehicles`, `silver_routes`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, hour, dayofweek

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
dl = spark.read.table('silver_deliveries')
veh = spark.read.table('silver_vehicles')
rt = spark.read.table('silver_routes')
print(f'deliveries={dl.count():,} vehicles={veh.count():,} routes={rt.count():,}')

required = {'delivery_id', 'vehicle_id', 'route_id', 'is_late', 'planned_duration_hrs'}
missing = required - set(dl.columns)
if missing:
    raise ValueError(f'silver_deliveries missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Join attributes + derive pre-departure features. EXCLUDE leakage columns.
ml_features = (
    dl.select(
        'delivery_id', 'vehicle_id', 'route_id', 'planned_departure',
        'planned_duration_hrs', 'distance_km', 'load_tonnes',
        col('is_late').alias('had_late'),
    )
    .join(veh.select('vehicle_id', 'vehicle_type', 'depot', 'capacity_tonnes', 'vehicle_age'),
          'vehicle_id', 'left')
    .join(rt.select('route_id', 'route_type', 'sla_hours', 'toll_cost_gbp'),
          'route_id', 'left')
    .withColumn('load_utilisation',
        when(col('capacity_tonnes') > 0, col('load_tonnes') / col('capacity_tonnes')).otherwise(0.0))
    .withColumn('departure_hour', hour('planned_departure'))
    .withColumn('departure_dow', dayofweek('planned_departure'))
    .withColumn('is_weekend', when(dayofweek('planned_departure').isin(1, 7), 1).otherwise(0))
    .withColumn('is_rush', when(hour('planned_departure').isin(7, 8, 9, 16, 17, 18, 19), 1).otherwise(0))
    .na.fill(0)
    .na.fill('unknown', subset=['vehicle_type', 'depot', 'route_type'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_late') == 1).count()
late_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} late rows '
        f'({late_rate:.2f}%). Check is_late typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | late rate {late_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')